# CARBAMYLATION ANALYSIS TOOL

This Jupyter notebook enables selecting and curating protein atomic structures for the analysis of their lysines. The aim, is to identify the subset of lysines that may be carbamylated. The pipeline is divided in the following steps.

* 1. [Data Gathering](#gather)
  - 1.1. [Mine UNIPROT database](#uniprot)
  - 1.2. [Mine PDB and AlphaFold databases](#pdb)
* 2. [Feature Extraction](#measure)
* 3. [Data Analysis](#analysis)

Further explanations about each of these steps are provided in the individual sections below.
Now, let's execute the cell below (press Shift+ENTER keys) to import all the Python packages required.

In [1]:
import os

#data handling
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", None, "display.max_columns", None)

#plotting
import plotly.express as px
import plotly.graph_objects as go
import nglview as nv
from ipywidgets import HBox, VBox

# atomic structures manipulation
import biobox as bb

#carbamylation analysis procedures
from uniprot import Uniprot
from protein import PDB
from measure import Measure
import postprocessing as postproc

C:\Users\xdzl45\Anaconda3\lib\site-packages\blessed\terminal.py:186: UserWarning: Failed to setupterm(kind='xterm-color'): Could not find terminal xterm-color
  warnings.warn(msg)


Before getting started, let's define an output directory. If none defined, the default name will be "result".

In [2]:
outdir="result"

## 1. Data Gathering  <a class="anchor" id="gather"></a>

### 1.1. Mine UNIPROT database  <a class="anchor" id="uniprot"></a>

Load data from UNIPROT database. The commands below can be launched multiple times, to compile a database saved in a pandas DataFrame (`Uniprot.df`).

In [ ]:
UP = Uniprot()
#UP.get_protein_data("P09167") # example: load structures for a single UNIPROT
UP.from_csv_file("inputs\\input_codes_4.csv") # add structures from a .csv file
UP.df

### 1.2. Mine PDB and AlphaFold databases  <a class="anchor" id="pdb"></a>

Load and curate all proteins described in a Uniprot DataFrame (`Uniprot.df`). Two optional parameters can be passed to the PDB constructor, `outdir` (output directory, "result" by default) and the largest acceptable gap in sequence (`gap`, 10 by default).

For each entry in the database:
* raw files are downloaded from the PDB in a subfolder within the output directory (defined by `PDB.raw_dir`, "conformations" by default)
* each raw file is cleaned: heteroatoms, hydrogens and non-protein, non-standard aminoacids are amended. Ions are preserved, as they may have important effect on the feature extraction phase
* different conformations (usually from NMR ensembles) are saved in different files
* if multiple aminoacid rotamers exist, multiple files are produced
* all produced files are patched, i.e. missing parts in the sequence are added using Modeller
* all files correctly patched are saved in a different subfolder in the output directory (defined by `PDB.curated_dir`, "curated" by default).

This process will generate a new DataFrame, `PDB.df`, containing information on all the structures that have been correctly curated.

<div class="alert alert-warning">
<b>Warning:</b> depending on how many proteins have to be downloaded and curated, this operation may take a long time. AlphaFold and PDB files not needing curation are processed in few seconds. Large multimers requiring curation can take up to 1 min. Multiple conformations in a single protein structure are individually curated.
</div>

In [ ]:
pdb = PDB(outdir=outdir, gap=10)
pdb.gather_proteins(UP.df)

Show the DataFrame reporting on structures successfully downloaded and curated, and save it into a CSV file. If no filename is given, file `proteins.csv` will be saved in the output directory. This CSV file can be later loaded with the method `PDB.load_state(filename)`. 

In [ ]:
pdb.df
pdb.save_state()

### 2. Feature Extraction  <a class="anchor" id="measure"></a>

For all structures correctly curated in the previous step, calculate pKa and Solvent Accessible Surface Area (SASA) and save final result into a file. If no filename is given, the file will be called `measures.csv`.

<div class="alert alert-warning">
<b>Warning:</b> calculations typically take 5-30 sec. per curated protein structure (depending on its size and number of lysines).
</div>

In [ ]:
M = Measure(pdb.df, outdir=outdir)
M.measure_dataframe()
M.save_state()

Print the final measures output, sorted by UNIPROT entry first, and then by resid.

In [ ]:
result = M.df.dropna()
result.sort_values(['Uniprot Entry', 'Resid'])

### 3. Data Analysis  <a class="anchor" id="analysis"></a>

<div class="alert alert-warning">
<b>Warning:</b> From here onwards, the code is not yet fully implemented. See the markdown comments as a guide for what needs to be done.</div>

The Analysis class is resonsible of interpreting data stored in the DataFrame `Measures.df`. Data is saved in `Analysis.df` and  automatically aggregated in `Analysis.df_aggregate`. Aggregation involves calculating min, max, mean and standard deviation for each feature (a single lysine may features multiple measurements in the input dataframe).
Here, the ground truth (i.e. known carbamylated lysines) can also be loaded. Ground truth is appended as an extra column to `Analysis.df` and `Analysis.df_aggregate`.

Start by initializing `Analysis` either via a previously saved logfile, or from a DataFrame in a `Measure` instance.

In [ ]:
if True: # is a Measures instance was produced above
    df = M.df
else: # if a logfile storing a Measures DataFrame calculated in a previous session 
    df = pd.read_csv("results\\results_2608\\results.csv")

# A = Analysis(df)

#TODO 1: Analysis should aggregate data according to the selection below (extract mean and std of each measure)
#entries = result.drop_duplicates(subset=["Uniprot Entry", "Resid"])[["Uniprot Entry", "Resid"]]
#entries
#
#TODO 2: Analysis should load experimental evidence (ground truth, i.e. what is carbamylated) into an Analysis class.
#        The class stores both aggregated and raw data. Ground truth appended as extra column.

Plot data on interactive plot. Gray points represent the aggregated data, featuring one point per uniprot code. Clicking on any point will show additional points in red. These points are the measures performed on every protein structure associated with the uniprot code selected by clicking.

In [ ]:
#ISSUE: hovering always looks as if the aggregate plot is selected, although focus items are clicked

df_data = M.df

def update_point(trace, points, selector):
          
    #print(trace, points)    
    
    # only work with gray points
    if len(points.point_inds) == 0:
        return  
    
    # gather uniprot and chain ID from full dataset clicked point
    idx = points.point_inds[0]
    my_uniprot = df_data.loc[idx, "Uniprot Entry"]
    my_resid = df_data.loc[idx, "Resid"]
    my_label = "%s(%i)"%(my_uniprot, my_resid)
    df_query = df.query('`Uniprot Entry` == "%s" and resid == %s'%(my_uniprot, my_resid))

    #plot data associated with selection
    #labels = ["UNIPROT: %s<br>resid: %i"%(df_query["Uniprot Entry"].values[i], df_query["resid"].values[i]) for i in range(len(df_query))]
    labels = ["PDB: %s"%(df_query["PDB Code"].values[i]) for i in range(len(df_query))]
    if len(f.data)>1:
        f.data = [f.data[0]]

    f.add_scatter(x=df_query["sasa"], y=df_query["pKa"],
                  mode='markers', showlegend=False, name=my_label,
                  text = labels, hovertemplate='%{text}<br>SASA: %{x:.2f}<br>pKa: %{y:.2f}')

    
    #identify clicked point and mark it differently (e.g. different colour or size)
    #c = list(scatter.marker.color)
    #for i in points.point_inds:
    #    c[i] = '#ffffff'
    #    with f.batch_update():
    #        scatter.marker.color = c

# create scatter plot
labels = ["UNIPROT: %s<br>resid: %i"%(df_data["Uniprot Entry"].values[i], df_data ["Resid"].values[i]) for i in range(len(df_data))]
f = go.FigureWidget([go.Scatter(x=df_data["sasa"], y=df_data["pKa"],
                                mode='markers', name="aggregate", showlegend=False, opacity=0.75,
                                text = labels, hovertemplate='%{text}<br>SASA: %{x:.2f}<br>pKa: %{y:.2f}'
                               )
                    ])

# add labels
f.update_layout(
    xaxis_title="SASA (A2)",
    yaxis_title="pKa")

#set axes properties
f.update_xaxes(range=[0, 130])
f.update_yaxes(range=[-2, 20])
f.update_xaxes(showspikes=True)
f.update_yaxes(showspikes=True)

#parameterize colours and triggers
scatter = f.data[0]
colors = ['#aaaaaa'] * len(df_data)
scatter.marker.color = colors
f.layout.hovermode = 'closest'
scatter.on_click(update_point)
f

Select desired uniprot entry and resid from the logfile, based on observation of the plot above.

In [ ]:
# EXAMPLE
uniprot = "P15531" #insert uniprot code here
resid = 85 #insert resid here

Display all lysine entries matching the uniprot code and resid provided above, and provide statistics (this is useful if more than one lysine conformation was analysed).

In [ ]:
df_query = df.query('`Uniprot Entry` == "%s" and resid == %s'%(uniprot, resid))
pKa = df_query["pKa"]
sasa = df_query["sasa"]

print("uniprot %s, resid %s\n%s measures\npKa %4.2f pm %4.2f [%4.2f %4.2f]\nsasa %4.2f pm %4.2f [%4.2f %4.2f] A2"%
      (uniprot, resid, len(pKa),
       np.mean(pKa), np.std(pKa), np.min(pKa), np.max(pKa),
       np.mean(sasa), np.std(sasa), np.min(sasa), np.max(sasa)))

df.query('`Uniprot Entry` == "%s" and resid == %s'%(uniprot, resid))

Select index of an entry of interest, to be visualized in 3D

In [ ]:
idx = 8699

Display the structure of interest

In [ ]:
resid = df.loc[idx, "resid"]
chain = df.loc[idx, "chain"]
PDB = "%s%s%s.pdb"%(folder_path, os.sep, df.loc[idx, "PDB Code"])
print("file: %s, chain: %s, resid: %s"%(PDB, chain, resid))

#u = mda.Universe(PDB)
#protein = u.select_atoms('protein')
#view = nv.show_mdanalysis(protein)
view

In [ ]:
# SCATCH AREA, EXAMPLE CODES FOR DIFFERENT USAGES #

# example: show data visible through hovering over points
#
#fig = px.scatter(df_datae, x="sasa", y="pKa", opacity=0.5,
#                 #error_x="sasa stdev", error_y="pKa stdev",
#                 hover_data=["PDB Code", "resid", "chain", "Uniprot Entry"])
#fig.show()

In [ ]:
import plotly.graph_objects as go
from ipywidgets import HBox, VBox

np.random.seed(1)

x = np.random.rand(100)
y = np.random.rand(100)

e = go.FigureWidget([go.Scatter(x=x, y=y, mode='markers')])#, go.Scatter(x=x, y=y, mode='markers')])
g = go.FigureWidget([go.Scatter(x=x, y=y, mode='markers')])#, go.Scatter(x=x, y=y, mode='markers')])

scattere = e.data[0]
colors = ['#a3a7e4'] * 100
scattere.marker.color = colors
scattere.marker.size = [10] * 100
e.layout.hovermode = 'closest'

scatterg = g.data[0]
colors = ['#a3a7e4'] * 100
scatterg.marker.color = colors
scatterg.marker.size = [10] * 100
g.layout.hovermode = 'closest'

# create our callback function
def update_pointe(trace, points, selector):
    c = list(scattere.marker.color)
    s = list(scattere.marker.size)
    for i in points.point_inds:
        c[i] = '#bae2be'
        s[i] = 20
        with e.batch_update():
            scattere.marker.color = c
            scattere.marker.size = s
            
# create our callback function
def update_pointg(trace, points, selector):
    c = list(scatterg.marker.color)
    s = list(scatterg.marker.size)
    for i in points.point_inds:
        c[i] = '#bae2be'
        s[i] = 20
        with g.batch_update():
            scatterg.marker.color = c
            scatterg.marker.size = s

scattere.on_click(update_pointg)
scatterg.on_click(update_pointe)

VBox([e,g])

In [ ]:
def set_opacity(opacity, size):
    scattere.marker.opacity = opacity
    scattere.marker.size = size

from ipywidgets import interactive
opacity_slider = interactive(set_opacity,
                            opacity=(0.0, 1.0, 0.01),
                            size=(1,10,0.25))

opacity_slider.children[0].layout.width = '400px'
opacity_slider.children[1].layout.width = '400px'

VBox([e,g, opacity_slider])